# 🏦 Task 3: Customer Churn Prediction (Bank Customers)
**Internship: DevelopersHub Corporation — Data Science & Analytics**

---
## 1. Introduction & Problem Statement
Identify bank customers likely to churn (leave the bank) using demographic and account data.

**Dataset:** Churn Modelling Dataset  
**Model:** Random Forest Classifier

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
sns.set_theme(style='whitegrid')
print('✅ Libraries imported')

## 2. Dataset Understanding

In [ ]:
df = pd.read_csv('churn_modelling.csv')
print('Shape:', df.shape)
df.head()

## 3. Data Cleaning & Preparation

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())

df.drop(columns=['RowNumber','CustomerId','Surname'], inplace=True)

# Encode categorical features
le_geo = LabelEncoder()
le_gender = LabelEncoder()
df['Geography'] = le_geo.fit_transform(df['Geography'])
df['Gender'] = le_gender.fit_transform(df['Gender'])
print('✅ Categorical encoding done (Geography, Gender)')

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))
df['Exited'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'])
axes[0].set_title('Churn Distribution')
sns.histplot(df['Age'], hue=df['Exited'], kde=True, ax=axes[1])
axes[1].set_title('Age vs Churn')
sns.boxplot(data=df, x='Exited', y='Balance', ax=axes[2])
axes[2].set_title('Balance vs Churn')
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150)
plt.show()

## 5. Model Training

In [ ]:
X = df.drop(columns=['Exited'])
y = df['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
print('✅ Model trained')

## 6. Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Stayed','Churned'], cmap='Reds')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 7. Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8,5))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print(importances)

## 8. Conclusion
- Age, Balance, and NumOfProducts are top churn drivers.
- Inactive members and customers in Germany show higher churn tendency.
- Random Forest captures non-linear feature interactions well for this task.